# Reto 03 — SOLUCION

No liberar hasta cerrar el modulo.

In [ ]:
crudo = [
    "V01;Laptop;2;3200.00",
    "V02; Mouse ;5;65,00",
    "V03;Monitor;tres;780.00",
    "V04;Teclado;4;120.00",
    "V05;Audifonos;;210.00",
    "V06;Webcam;3",
    "V07;Silla;-2;640.00",
    "V08;Impresora;1;950.00",
]

## 1 — `limpiar_numero`

Una funcion, un trabajo. Convierte y punto. **No** captura el error: si el texto no es
un numero, quien llama es quien sabe si eso es fatal o si hay que saltarse la fila.

In [ ]:
def limpiar_numero(texto):
    """Convierte un texto a float aceptando coma decimal. Lanza ValueError si no puede."""
    return float(texto.strip().replace(",", "."))


assert limpiar_numero(" 3200.00 ") == 3200.0
assert limpiar_numero("65,00") == 65.0
print("OK ejercicio 1")

## 2 — `parsear_linea`

Valida en orden: estructura -> tipos -> reglas de negocio.
Cada `raise` dice **que** dato fallo, no solo que hubo un error.

In [ ]:
def parsear_linea(linea):
    """Convierte 'ID;producto;cantidad;precio' en dict. Lanza ValueError con el detalle."""
    campos = linea.split(";")
    if len(campos) != 4:
        raise ValueError(f"se esperaban 4 campos, llegaron {len(campos)}")

    id_venta, producto, cant_txt, precio_txt = (c.strip() for c in campos)

    try:
        cantidad = limpiar_numero(cant_txt)
    except ValueError:
        raise ValueError(f"cantidad invalida: {cant_txt!r}")
    try:
        precio = limpiar_numero(precio_txt)
    except ValueError:
        raise ValueError(f"precio invalido: {precio_txt!r}")

    if cantidad < 0:
        raise ValueError(f"cantidad negativa: {cantidad}")

    return {"id": id_venta, "producto": producto, "cantidad": cantidad,
            "precio": precio, "importe": cantidad * precio}


d = parsear_linea("V01;Laptop;2;3200.00")
assert d["importe"] == 6400.0
assert parsear_linea("V02; Mouse ;5;65,00")["producto"] == "Mouse"
print("OK ejercicio 2")

## 3 — `procesar`

El `try/except` va **aqui**, en el bucle, no dentro de `parsear_linea`. Ese es el punto
del ejercicio: la funcion que sabe parsear no sabe si un fallo es tolerable; la que
recorre el lote, si.

In [ ]:
def procesar(lineas):
    """Parsea todas las lineas. Devuelve (validas, errores) sin detenerse ante un fallo."""
    validas, errores = [], []
    for linea in lineas:
        try:
            validas.append(parsear_linea(linea))
        except ValueError as e:
            errores.append((linea, str(e)))
    return validas, errores


validas, errores = procesar(crudo)
assert (len(validas), len(errores)) == (4, 4)
print(f"OK ejercicio 3 — {len(validas)} validas, {len(errores)} errores")

## 4 — `reporte`

In [ ]:
def reporte(validas, errores):
    """Imprime el resumen del lote y el detalle de las lineas rechazadas."""
    total = sum(v["importe"] for v in validas)
    print("=== RESUMEN ===")
    print(f"Procesadas : {len(validas) + len(errores)}")
    print(f"Validas    : {len(validas)}")
    print(f"Con error  : {len(errores)}")
    print(f"Facturado  : S/ {total:,.2f}")
    if errores:
        print("\n=== LINEAS RECHAZADAS ===")
        for linea, motivo in errores:
            print(f"  {linea:32}  ->  {motivo}")


reporte(validas, errores)

## 5 — `resumir_por`

Generica por el nombre del campo. `v[campo]` lanza `KeyError` solo, pero el mensaje
por defecto no dice cuales SI existen — por eso lo relanzamos con contexto.

In [ ]:
def resumir_por(validas, campo):
    """Agrupa por `campo` y suma importes. KeyError si el campo no existe."""
    if validas and campo not in validas[0]:
        raise KeyError(f"campo {campo!r} inexistente; disponibles: {list(validas[0])}")
    out = {}
    for v in validas:
        out[v[campo]] = out.get(v[campo], 0) + v["importe"]
    return out


r = resumir_por(validas, "producto")
assert r["Laptop"] == 6400.0
print("OK ejercicio 5:", r)

---
### Errores tipicos al corregir

- **`except Exception:` o `except:` pelado.** Se traga el `KeyboardInterrupt` y los typos del propio alumno. Es el error mas comun y el mas caro en produccion.
- **`try/except` dentro de `parsear_linea` devolviendo `None`.** Entonces `procesar` tiene que preguntar `if resultado is None`, y se pierde el motivo del fallo.
- **`print` dentro de las funciones de calculo.** Imposible de testear y de reusar.
- **Repetir `float(x.strip().replace(...))` en vez de llamar a `limpiar_numero`.** Cuando cambie la regla del decimal hay que tocarlo en tres sitios.
- **No validar la cantidad negativa** porque `float("-2")` no falla. El error de tipo y el error de negocio son cosas distintas.
- **Mensajes tipo `"error en la linea"`.** Inutiles a las 2am. El mensaje debe traer el valor culpable.